In [27]:
from model import run_exp
from model import default_params as params
import utils as utl
from brian2 import Hz

config = {
    'path_res'  : './results/example',                              # directory to store results
    'path_comp' : './2023_03_23_completeness_630_final.csv',        # csv of the complete list of Flywire neurons
    'path_con'  : './2023_03_23_connectivity_630_final.parquet',    # connectivity data
    'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
}

In [28]:
import sys; print(sys.executable)

/home/connork/miniforge3/envs/brian2/bin/python


# Introduction
## Underlying connectivity data
The connectivity of the fly brain is stored in the folowing files:
- neurons present: `config['path_comp']`
- connectivity between neurons: `config['path_con]`

## Model parameters
The equation and constants for the leaky integrate and fire model are defined 
in the dictionary `default_params` in the beginning of the file `model.py`:

```
default_params = {
    # trials
    't_run'     : 1000 * ms,              # duration of trial
    'n_run'     : 30,                     # number of runs

    'v_0'       : -52 * mV,               # resting potential
    'v_rst'     : -52 * mV,               # reset potential after spike
    [...]
```
We can also change values
and pass the modified dictionary to the model (see Experiment 1).

## Addressing neurons
Here, we want to stimulate some sugar-sensing neurons in the right hemisphere.
The neurons of interest are defined via their flywire IDs:

In [29]:
config['path_res'] = './results/mine'

In [30]:
neu_sugar = [
    720575940624963786,
    720575940630233916,
    720575940637568838,
    720575940638202345,
    720575940617000768,
    720575940630797113,
    720575940632889389,
    720575940621754367,
    720575940621502051,
    720575940640649691,
    720575940639332736,
    720575940616885538,
    720575940639198653,
    720575940620900446,
    720575940617937543,
    720575940632425919,
    720575940633143833,
    720575940612670570,
    720575940628853239,
    720575940629176663,
    720575940611875570,
]

For an easier identification, we define also a mapping from the flywire IDs to custom 
names. The above neurons are calles `sugar_1`, `sugar_2` etc:

In [31]:
flyid2name = { f: f'sugar_{i+1}' for i, f in enumerate(neu_sugar) }
flyid2name

{720575940624963786: 'sugar_1',
 720575940630233916: 'sugar_2',
 720575940637568838: 'sugar_3',
 720575940638202345: 'sugar_4',
 720575940617000768: 'sugar_5',
 720575940630797113: 'sugar_6',
 720575940632889389: 'sugar_7',
 720575940621754367: 'sugar_8',
 720575940621502051: 'sugar_9',
 720575940640649691: 'sugar_10',
 720575940639332736: 'sugar_11',
 720575940616885538: 'sugar_12',
 720575940639198653: 'sugar_13',
 720575940620900446: 'sugar_14',
 720575940617937543: 'sugar_15',
 720575940632425919: 'sugar_16',
 720575940633143833: 'sugar_17',
 720575940612670570: 'sugar_18',
 720575940628853239: 'sugar_19',
 720575940629176663: 'sugar_20',
 720575940611875570: 'sugar_21'}

# Running simulations
## Activating a set of neurons
To run a simulation exciting these nerons we have to call `run_exp` supplying the following:
- unique name for the simulation: `exp_name`
- a list of neurons we want to stimulate: `neu_sugar`
- the connectivity data: `config['path_comp']` and `config['path_con]`
- path to store the output: `config['path_res']`
- number of CPU cores use: `config['n_procs]`

Note that running this on Google Colab can take roughly 20 minutes; it is substantially faster on a local install, depending on the number of CPU cores. By default, the neurons are excited at 200 Hz.

In [32]:
# activate sugar sensing neurons
run_exp(exp_name='sugarR', neu_exc=neu_sugar, **config)

>>> Skipping experiment sugarR because results/mine/sugarR.parquet exists and force_overwrite = False


The `.parquet` file created during a simulation contains all spikes events of all neurons in the model.
We load the data again from disk by passing a list of result files to the `utl.load_exps` function.

We can see from the size of the dataframe
that more than 400 000 spikes were generated by activating the sugar neurons (30 trials, 1 s each).

In [33]:
# load data from disk
df_spike = utl.load_exps([ './results/example/sugarR.parquet' ])
df_spike

,t,trial,flywire_id,exp_name
0,0.2562,0,720575940605513649,sugarR
1,0.3607,0,720575940605513649,sugarR
2,0.6044,0,720575940605513649,sugarR
3,0.7128,0,720575940605513649,sugarR
4,0.8627,0,720575940605513649,sugarR
...,...,...,...,...
511561,0.9133,29,720575940660229505,sugarR
511562,0.9245,29,720575940660229505,sugarR
511563,0.9437,29,720575940660229505,sugarR
511564,0.9687,29,720575940660229505,sugarR


The spike times can be converted to spike rates [Hz] via `utl.get_rate`, which requires the duration of each trial.
`utl.get_rate` returns `pandas.DataFrame` objects:
1. spike rate for each neuron (rows) in each experiment (column): `df_rate`
2. standard deviation of rate across trials: `df_rate_std`

For convenience, we can optionally pass the `flyid2name` dictionary to `utl.get_rate` in order to convert flywire IDs into
meaningful names.

We can see that only about 400 neurons show activity during the simulations.

In [34]:
# calculate spike rate and standard deviation
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
# sort by spike rate
df_rate.sort_values('sugarR', ascending=False)

exp_name,name,sugarR
flyid,,
720575940637568838,sugar_3,202.066667
720575940621502051,sugar_9,200.200000
720575940639198653,sugar_13,199.966667
720575940617937543,sugar_15,199.400000
720575940633143833,sugar_17,199.033333
...,...,...
720575940626919144,,0.033333
720575940614465478,,0.033333
720575940612866410,,0.033333


## Change stimulation frequency

We want to change the frequency of the stimulation of the sugar neurons.
To do so we modify the value for `r_poi` in the `default_params` dictionary and pass the altered dictionary to the `run_exp` function.

Note: Since physical quantities in `brian2` have to have the correct unit, we also need the `brian2.Hz` object 
to define a frequency.

In [35]:
# run with different frequency
params['r_poi'] = 100 * Hz

run_exp(exp_name='sugarR_100Hz', neu_exc=neu_sugar, params=params, **config)

>>> Skipping experiment sugarR_100Hz because results/mine/sugarR_100Hz.parquet exists and force_overwrite = False


We load the results via the `utl.load_exps` function and convert the spike events to rates with `utl.get_rate`

In [36]:
ps = [
    './results/example/sugarR.parquet',
    './results/example/sugarR_100Hz.parquet',
]

df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'], flyid2name=flyid2name)
df_rate.sort_values('sugarR_100Hz', ascending=False, inplace=True)
df_rate

exp_name,name,sugarR,sugarR_100Hz
flyid,,,
720575940622695448,,156.766667,114.266667
720575940621754367,sugar_8,194.100000,102.566667
720575940617937543,sugar_15,199.400000,101.733333
720575940629888530,,146.500000,101.700000
720575940629176663,sugar_20,191.866667,101.266667
...,...,...,...
720575940639813365,,0.066667,NaN
720575940643551392,,0.733333,NaN
720575940644666148,,1.133333,NaN


## Silencing neurons
We want to silence the most active neurons individually to see how that changes the activity patterns.
We do so by passing the neuron IDs we want to silence as a list `run_exp` via the `neu_slnc` argument.
In the following example, we are silencing a single neuron `[ i ]` while exciting the sugar neurons `neu_sugar`. 
We can then investigate how silencing of each individual neuron affects the firing rate of a given neuron, say, MN9. 

In [37]:
#First, let's check on the MN9 firing rate when no neurons are silenced.
id_mn9 = 720575940660219265 #id for MN9
x = df_rate.loc[id_mn9, "sugarR_100Hz"]
print(f'Rate for neuron {id_mn9} is {x}')

Rate for neuron 720575940660219265 is 67.03333333333333


In [38]:
# IDs of 3 most active neurons. These neurons are all sugar-sensing neurons.
ids = df_rate.sort_values('sugarR_100Hz', ascending=False).index[:3]

for i in ids:
    run_exp(exp_name=f'sugarR-{i}', neu_exc=neu_sugar, neu_slnc=[ i ], params=params, **config)

>>> Skipping experiment sugarR-720575940622695448 because results/mine/sugarR-720575940622695448.parquet exists and force_overwrite = False
>>> Skipping experiment sugarR-720575940621754367 because results/mine/sugarR-720575940621754367.parquet exists and force_overwrite = False
>>> Skipping experiment sugarR-720575940617937543 because results/mine/sugarR-720575940617937543.parquet exists and force_overwrite = False


In [39]:
# output files
ps = [ f'./results/example/sugarR-{i}.parquet' for i in ids ]

# calculate spike rate and sort
df_spike = utl.load_exps(ps)
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])
df_rate.loc[id_mn9, :].sort_values(ascending=True)

exp_name
sugarR-720575940617937543    63.266667
sugarR-720575940621754367    66.900000
sugarR-720575940622695448    71.166667
Name: 720575940660219265, dtype: float64

In [40]:
config['path_res'] = './results/mine'
df_spike = utl.load_exps(['./results/mine/sugarR_100Hz.parquet'])
df_rate, df_rate_std = utl.get_rate(df_spike, t_run=params['t_run'], n_run=params['n_run'])
print('MN9:', df_rate.loc[id_mn9, 'sugarR_100Hz'], '+/-', df_rate_std.loc[id_mn9, 'sugarR_100Hz'])
print('neurons fired:', len(df_rate))

MN9: 11.433333333333334 +/- 25.642001135290162
neurons fired: 390


In [41]:
print(params)

{'t_run': 1. * second, 'n_run': 30, 'v_0': -52. * mvolt, 'v_rst': -52. * mvolt, 'v_th': -45. * mvolt, 't_mbr': 20. * msecond, 'tau': 5. * msecond, 't_rfc': 2.2 * msecond, 't_dly': 1.8 * msecond, 'w_syn': 275. * uvolt, 'r_poi': 100. * hertz, 'r_poi2': 0. * hertz, 'f_poi': 250, 'eqs': '\ndv/dt = (v_0 - v + g) / t_mbr : volt (unless refractory)\ndg/dt = -g / tau               : volt (unless refractory) \nrfc                            : second\n', 'eq_th': 'v > v_th', 'eq_rst': 'v = v_rst; w = 0; g = 0 * mV'}


In [42]:
RATE_KEY = '...'   # fill in
rates = [10, 25, 50, 100, 200]
for r in rates:
    p = dict(params); p[RATE_KEY] = r * Hz; p['n_run'] = 5
    run_exp(exp_name=f'sugarR_{r}Hz', neu_exc=neu_sugar, neu_slnc=[], params=p, **config)

>>> Skipping experiment sugarR_10Hz because results/mine/sugarR_10Hz.parquet exists and force_overwrite = False
>>> Skipping experiment sugarR_25Hz because results/mine/sugarR_25Hz.parquet exists and force_overwrite = False
>>> Skipping experiment sugarR_50Hz because results/mine/sugarR_50Hz.parquet exists and force_overwrite = False
>>> Skipping experiment sugarR_100Hz because results/mine/sugarR_100Hz.parquet exists and force_overwrite = False
>>> Skipping experiment sugarR_200Hz because results/mine/sugarR_200Hz.parquet exists and force_overwrite = False


In [43]:
ps = [f'./results/mine/sugarR_{r}Hz.parquet' for r in rates]
df_spike = utl.load_exps(ps)
df_rate, _ = utl.get_rate(df_spike, t_run=params['t_run'], n_run=5)
df_rate.loc[id_mn9, :]

exp_name
sugarR_100Hz    68.6
sugarR_10Hz     70.2
sugarR_200Hz    66.0
sugarR_25Hz     66.2
sugarR_50Hz     67.6
Name: 720575940660219265, dtype: float64

In [44]:
df_rate.loc[neu_sugar, :].mean()

exp_name
sugarR_100Hz    100.628571
sugarR_10Hz      99.057143
sugarR_200Hz     99.009524
sugarR_25Hz      97.123810
sugarR_50Hz      98.666667
dtype: float64

In [47]:
!grep -n "poi\|Hz\|rate" model.py | head -30

6:from brian2 import mV, ms, Hz
38:    # Default activation rates 
39:    'r_poi'     : 150*Hz,                 # default rate of the Poisson inputs
40:    'r_poi2'    :   0*Hz,                 # default rate of a 2nd class of Poisson inputs
41:    'f_poi'     : 250,                    # scaling factor for Poisson synapse; 250 is sufficient to cause spiking
43:    # equations for neurons               # alpha synapse https://doi.org/10.1017/CBO9780511815706; See https://brian2.readthedocs.io/en/stable/user/converting_from_integrated_form.html
58:def poi(neu, exc, exc2, params):
61:    For each neuron in 'names' a PoissonInput is generated and 
71:        Indices of neurons for which to create Poisson input with `r_poi`
77:    pois : list
83:    pois = []
89:            rate=params['r_poi'], 
90:            weight=params['w_syn']*params['f_poi']
93:        pois.append(p)
100:            rate=params['r_poi2'], 
101:            weight=params['w_syn']*params['f_poi']
104:        pois.appen